# FairFace Pipeline Evaluation

This notebook tests whether the complete face-detection, skin-sampling,
and recommendation pipeline finishes successfully on a balanced sample
of 140 adult FairFace images (20 from each of seven labelled groups).

**Important:** FairFace does not provide correct foundation labels. This
notebook measures pipeline completion, not shade-match accuracy. Group
rates are descriptive diagnostics and are not proof of fairness.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/amafteeva/Predictive-beauty-analysis-using-computer-vision-and-machine-learning-.git"


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


project_root = find_project_root(Path.cwd())


    project_root = Path("/content/foundation-shade-recommender")
    if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(project_root)], check=True)

if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

os.chdir(project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Project root:", project_root)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from IPython.display import display

from foundation_matcher.config import FAIRFACE_RACE_NAMES
from foundation_matcher.data import load_foundation_catalog
from foundation_matcher.evaluation import (
    evaluate_face_pipeline,
    select_balanced_adult_samples,
    simulate_colour_robustness,
    summarize_group_completion,
)
from foundation_matcher.face import (
    create_face_landmarker,
    download_face_landmarker,
)
from foundation_matcher.visualization import (
    plot_group_completion,
    plot_previews,
)


## 2. Build one reproducible 140-image sample

The original notebook selected 35 images and later asserted that the same
object contained 140. This version selects 20 images per group once and
verifies the final total before evaluation.


In [ ]:
products = load_foundation_catalog()
model_path = download_face_landmarker("models/face_landmarker.task")

fairface_stream = load_dataset(
    "HuggingFaceM4/FairFace",
    "0.25",
    split="validation",
    streaming=True,
).shuffle(seed=42, buffer_size=2_000)

balanced_samples = select_balanced_adult_samples(
    fairface_stream,
    samples_per_group=20,
)

assert len(balanced_samples) == 140
print("Balanced evaluation images:", len(balanced_samples))


## 3. Run the complete pipeline

A run is successful only when a face is detected, skin colour is
extracted, and five recommendations are returned. Individual failures
are recorded instead of stopping the whole experiment.


In [ ]:
landmarker = create_face_landmarker(model_path)
try:
    evaluation, previews = evaluate_face_pipeline(
        balanced_samples,
        FAIRFACE_RACE_NAMES,
        landmarker,
        products,
        image_directory="outputs/fairface_test_140",
        top_n=5,
        preview_limit=14,
    )
finally:
    landmarker.close()

overall_completion = evaluation["pipeline_success"].mean()
print(f"Successful runs: {evaluation['pipeline_success'].sum()}/{len(evaluation)}")
print(f"Overall pipeline completion: {overall_completion:.1%}")

output_path = Path("outputs/tables/fairface_evaluation_140.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
evaluation.to_csv(output_path, index=False)
print("Saved:", output_path)


## 4. Inspect completion by group and failure reason


In [ ]:
group_summary = summarize_group_completion(evaluation)
display(group_summary.style.format({"success_rate": "{:.1f}%"}))

plot_group_completion(group_summary)
plt.show()


In [ ]:
failures = evaluation.loc[
    ~evaluation["pipeline_success"],
    ["image_id", "group", "error"],
]

print("Failed runs:", len(failures))
display(failures if not failures.empty else pd.DataFrame({"status": ["No failures"]}))

if previews:
    plot_previews(previews)
    plt.show()


## 5. Simulated colour robustness

This second experiment starts with a catalogue LAB colour, adds a small
synthetic perturbation, and checks what the recommender retrieves.

Exact-row recovery is not the same as perceptual quality because different
brands can share identical or nearly identical digital colours. Delta E
thresholds are therefore reported separately.


In [ ]:
robustness = simulate_colour_robustness(
    products,
    number_of_tests=500,
    random_state=42,
)

robustness_table = pd.Series(robustness, name="value").to_frame()
display(robustness_table.round(3))


## 6. What this evaluation can support

**Supported conclusion:** report the overall and group-specific pipeline
completion rates for this sample, with failure reasons and sample sizes.

**Unsupported conclusion:** do not call these results match accuracy,
precision, recall, or F1. Those metrics require ground-truth foundation
labels. A stronger study would collect professionally labelled matches,
calibrated images, repeated lighting conditions, and enough participants
for uncertainty estimates and subgroup analysis.
